# Gold Set Sampling — XSAMSum (zh) BART Baseline

Randomly sample a 50-example gold set from the full test set, then extract the corresponding dialogues from the original `test.json`:

1. Load `pair_scores_zh_XSAMSum_bart.csv`
2. Randomly sample 50 rows from the **full** test set with `random_state=42`
3. Look up the matching entries in `test.json` and emit them with `summary_de` removed

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42

INPUT_PATH      = "pair_scores_zh_XSAMSum_bart.csv"
TEST_JSON_PATH  = "test.json"                          # original XSAMSum test split
OUTPUT_CSV      = "gold_set_50_zh_XSAMSum_bart.csv"
OUTPUT_JSON     = "gold_set_50_zh_XSAMSum_bart.json"   # dialogues + retained summary fields

N_GOLD = 50

## 1. Load scores

In [2]:
df = pd.read_csv(INPUT_PATH, index_col=0)
print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
df.head()

Loaded 819 rows
Columns: ['reference', 'prediction', 'rouge1', 'rouge2', 'rougeL', 'bs_f1_raw']


,reference,prediction,rouge1,rouge2,rougeL,bs_f1_raw
0,汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。,汉娜不太了解他 阿曼达建议汉娜发短信给他,28.57,7.69,28.57,67.52
1,埃里克和罗伯要在youtube上看一场单口相声。,Eric和Rob正在看一个俄国喜剧演员的脱口秀,21.05,0.00,21.05,63.59
2,莱尼无法决定买哪条裤子。鲍勃就此给莱尼提了些建议。莱尼听了他的建议，选了质量最好的裤子。,Lenny会买两条黑裤子和一条紫色的,15.79,0.00,15.79,58.65
3,艾玛很快就会回家，而且她会告诉威尔。,Emma今晚不想做晚饭 她马上回家 Will会来接她,24.00,8.70,24.00,63.13
4,简在华沙，她和奥利有个聚会。她把重要的日子忘了，本来他们周五会共进午餐。但是奥利无意间给简打...,简从摩洛哥回来了 奥利和简周五下午6点会合吃午餐,32.26,10.00,22.58,68.02


In [3]:
df[["rougeL", "bs_f1_raw"]].describe()

,rougeL,bs_f1_raw
count,819.000000,819.000000
mean,30.714799,71.139841
std,15.356160,7.871887
min,0.000000,44.860000
25%,20.645000,65.755000
50%,28.570000,71.150000
75%,38.890000,75.985000
max,100.000000,97.520000


## 2. Random sample of 50 with fixed seed

In [4]:
assert len(df) >= N_GOLD, f"Test set has only {len(df)} rows"

gold = df.sample(n=N_GOLD, random_state=SEED).sort_index()
print(f"Sampled {len(gold)} examples from {len(df)} (seed={SEED})")
gold[["rougeL", "bs_f1_raw"]].describe()

Sampled 50 examples from 819 (seed=42)


,rougeL,bs_f1_raw
count,50.0000,50.000000
mean,34.6106,73.322000
std,14.9861,6.645837
min,6.6700,56.900000
25%,23.8975,69.865000
50%,33.8100,73.145000
75%,44.2000,77.102500
max,65.2200,85.540000


## 3. Sanity check — distribution of gold set vs. full set

In [5]:
pd.DataFrame({
    "full":   [df["rougeL"].mean(),   df["bs_f1_raw"].mean()],
    "gold50": [gold["rougeL"].mean(), gold["bs_f1_raw"].mean()],
}, index=["ROUGE-L mean", "BERTScore-F1 mean"]).round(2)

,full,gold50
ROUGE-L mean,30.71,34.61
BERTScore-F1 mean,71.14,73.32


## 4. Extract corresponding dialogues from `test.json`

The CSV's row index aligns with the position of each example in `test.json`. We:

1. Load `test.json`
2. Verify alignment by checking that `test_data[i]["summary_zh"]` matches the CSV's `reference` for each sampled row
3. Pull the 50 selected entries and drop `summary_de` from each

In [6]:
with open(TEST_JSON_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Loaded {len(test_data)} entries from {TEST_JSON_PATH}")
print(f"Fields in first entry: {list(test_data[0].keys())}")

Loaded 819 entries from test.json
Fields in first entry: ['dialogue', 'summary', 'summary_de', 'summary_zh']


In [7]:
# Verify CSV row index aligns with test.json position (spot-check on the gold sample)
mismatches = []
for i in gold.index:
    if test_data[i].get("summary_zh") != gold.loc[i, "reference"]:
        mismatches.append(i)

if mismatches:
    print(f"WARNING: {len(mismatches)} alignment mismatches at indices {mismatches[:5]}...")
    print("Index alignment between CSV and test.json may be off — investigate before proceeding.")
else:
    print(f"All {len(gold)} gold-set rows align with test.json by index ✓")

All 50 gold-set rows align with test.json by index ✓


In [8]:
# Build the gold dialogues, dropping summary_de
DROP_FIELDS = {"summary_de"}

gold_dialogues = []
for i in gold.index:
    entry = {k: v for k, v in test_data[i].items() if k not in DROP_FIELDS}
    # Track the original test.json index for traceability
    entry = {"test_index": int(i), **entry}
    gold_dialogues.append(entry)

print(f"Built {len(gold_dialogues)} gold-set entries")
print(f"Fields per entry: {list(gold_dialogues[0].keys())}")
gold_dialogues[0]

Built 50 gold-set entries
Fields per entry: ['test_index', 'dialogue', 'summary', 'summary_zh']


{'test_index': 23,
 'dialogue': "Anne: You were right, he was lying to me :/\nIrene: Oh no, what happened?\nJane: who? that Mark guy?\nAnne: yeah, he told me he's 30, today I saw his passport - he's 40\nIrene: You sure it's so important?\nAnne: he lied to me Irene",
 'summary': 'Mark lied to Anne about his age. Mark is 40.',
 'summary_zh': '马克向安妮隐瞒了自己的年龄。他40岁了。'}

## 5. Save

In [9]:
# Per-sample scores (CSV)
gold.to_csv(OUTPUT_CSV)
print(f"Wrote scores for {len(gold)} rows to {OUTPUT_CSV}")

# Dialogues with retained summary fields (JSON)
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(gold_dialogues, f, ensure_ascii=False, indent=2)
print(f"Wrote {len(gold_dialogues)} dialogues to {OUTPUT_JSON}")

Wrote scores for 50 rows to gold_set_50_zh_XSAMSum_bart.csv
Wrote 50 dialogues to gold_set_50_zh_XSAMSum_bart.json
